## Batch Inference

### Purpose

This notebook runs batch scoring using the approved `Champion` model.

It demonstrates multiple patterns for applying a registered model to large datasets and writing predictions to persistent storage.

Use this notebook when:
- Real-time inference is not required
- Scoring large historical datasets
- Running scheduled or pipeline-based inference jobs

### Preconditions

Before running this notebook:

- The model must be registered in Unity Catalog
- The `Champion` alias must be assigned
- Input data must be accessible as a table or DataFrame
- The model signature must align with the scoring dataset schema

### What This Notebook Demonstrates

- Loading the `Champion` model from Unity Catalog
- Applying the model in batch
- Writing predictions to a Delta table
- Multiple inference execution patterns

All approaches reference the `Champion` alias to maintain controlled promotion flow.

In [0]:
%pip install -q threadpoolctl>=3.5.0 xgboost
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

In [0]:
# Widget parameters for job orchestration
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("model_name", "pyfunc_xgb", "Model Name")
dbutils.widgets.text("input_table", "", "Input Table (leave empty for default)")
dbutils.widgets.text("output_table", "", "Output Table (leave empty for default)")

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
model_name = dbutils.widgets.get("model_name")
input_table = dbutils.widgets.get("input_table") or f"{catalog}.{schema}.iris_features"
output_table = dbutils.widgets.get("output_table") or f"{catalog}.{schema}.iris_predictions"

# Construct registered model name
registered_model_name = f"{catalog}.{schema}.{model_name}"

## Run batch inference

1. Loads the model using:

    - `catalog_name`
    - `schema_name`
    - `model_name`

    The `Champion` alias is resolved at runtime.

2. Batch score using Spark UDF. See `_inference_reference.ipynb` for other batch options.

3. After scoring:

    - Persist predictions to a Delta table
    - Include:
    - Prediction output
    - Model version
    - Timestamp
    - Optional evaluation metadata

    Ensure downstream consumers understand prediction schema.

**Operational Considerations**

When productionising batch inference:

- Schedule via Jobs or DABS
- Monitor execution success
- Track model version used
- Consider drift monitoring separately
- Avoid hardcoding model version numbers

Always reference the `Champion` alias unless performing controlled backtesting.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

champion_mv = client.get_model_version_by_alias(registered_model_name, "Champion")
champion_version = int(champion_mv.version)
if champion_mv.tags.get("approval") != "approved":
    raise ValueError(f"Champion model version {champion_version} is not approved")

model_uri = f"models:/{registered_model_name}@Champion"
model_info = mlflow.models.get_model_info(model_uri)
feature_cols = [inp.name for inp in model_info.signature.inputs]
id_cols = ["id"]

input_df = spark.table(input_table)
model_udf = mlflow.pyfunc.spark_udf(spark=spark, model_uri=model_uri, result_type="long", env_manager="virtualenv")
input_df = input_df.repartition(spark.sparkContext.defaultParallelism * 2)

scored_df = input_df.withColumn("prediction", model_udf(*[F.col(c) for c in feature_cols])).select(*id_cols, "prediction")
scored_df = scored_df.withColumn("model_version", F.lit(champion_version)).withColumn("prediction_timestamp", F.current_timestamp().cast("string"))
scored_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)
num_predictions = scored_df.count()

dbutils.jobs.taskValues.set(key="output_table", value=output_table)
dbutils.jobs.taskValues.set(key="num_predictions", value=str(num_predictions))
dbutils.jobs.taskValues.set(key="model_version", value=str(champion_version))

print(f"Batch inference completed: {registered_model_name} v{champion_version} -> {output_table} ({num_predictions} rows)")